## 📊 1. Título e Contexto

Este projeto tem como objetivo prever se um cliente irá aderir a um depósito a prazo, com base em dados provenientes de campanhas de marketing bancário.

O dataset utilizado é o **Bank Marketing**, disponibilizado pelo *UCI Machine Learning Repository*, amplamente utilizado em estudos de classificação.

O problema abordado é de **classificação binária**, no qual o modelo deve prever se o cliente irá aderir (*sim*) ou não (*não*) ao produto ofertado.

## 2. Importação e Carga dos Dados

Nesta etapa, o dataset é carregado diretamente a partir de uma URL pública, garantindo **reprodutibilidade** e permitindo que o notebook seja executado sem a necessidade de configurações adicionais no ambiente.

A utilização de fontes externas via URL também assegura que qualquer usuário consiga reproduzir os resultados de forma consistente, especialmente no ambiente do Google Colab.

In [1]:
import pandas as pd
import numpy as np

url = "https://archive.ics.uci.edu/static/public/222/data.csv"
df = pd.read_csv(url)

df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no


## 3. Análise Exploratória dos Dados (EDA)

Nesta etapa, é realizada uma análise exploratória dos dados com o objetivo de compreender a estrutura do dataset, identificar os tipos de variáveis e verificar a presença de valores ausentes.

Essa análise inicial é fundamental para orientar as etapas de pré-processamento e modelagem, garantindo maior qualidade nos dados utilizados.

A partir da análise, observa-se que o dataset possui variáveis numéricas e categóricas, além da presença de valores ausentes em algumas colunas, como **job**, **education**, **contact** e **poutcome**.

Esses valores ausentes deverão ser tratados nas próximas etapas antes da aplicação dos modelos de machine learning.

In [2]:
df.info()
df.describe()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   age          45211 non-null  int64 
 1   job          44923 non-null  object
 2   marital      45211 non-null  object
 3   education    43354 non-null  object
 4   default      45211 non-null  object
 5   balance      45211 non-null  int64 
 6   housing      45211 non-null  object
 7   loan         45211 non-null  object
 8   contact      32191 non-null  object
 9   day_of_week  45211 non-null  int64 
 10  month        45211 non-null  object
 11  duration     45211 non-null  int64 
 12  campaign     45211 non-null  int64 
 13  pdays        45211 non-null  int64 
 14  previous     45211 non-null  int64 
 15  poutcome     8252 non-null   object
 16  y            45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB


,0
age,0
job,288
marital,0
education,1857
default,0
balance,0
housing,0
loan,0
contact,13020
day_of_week,0


## 4. Tratamento de Dados

Nesta etapa, são realizados os ajustes necessários para preparar os dados para a aplicação dos algoritmos de machine learning.

Inicialmente, é feita a padronização dos nomes das colunas, removendo possíveis espaços em branco que possam causar inconsistências no processamento.

Em seguida, os valores ausentes são tratados, sendo substituídos por uma categoria padrão (**"unknown"**), garantindo que não haja perda de dados.

A variável alvo (**y**) é convertida para formato numérico, onde:
- **1** representa "sim"
- **0** representa "não"

Posteriormente, os dados são separados em:
- **X (features)**: variáveis explicativas
- **y (target)**: variável alvo

Por fim, é aplicado o **One-Hot Encoding** nas variáveis categóricas, transformando-as em formato numérico, conforme exigido pelos algoritmos de machine learning.

In [3]:
import pandas as pd

# limpar nomes das colunas
df.columns = df.columns.str.strip()

# tratar valores nulos
df = df.fillna("unknown")

# converter a variável alvo
df["y"] = df["y"].map({"yes": 1, "no": 0})

# separar features e target
X = df.drop("y", axis=1)
y = df["y"]

# aplicar one-hot encoding nas variáveis categóricas
X = pd.get_dummies(X, drop_first=True)

print("X shape:", X.shape)
print("y shape:", y.shape)

print('y' in df.columns)

X shape: (45211, 42)
y shape: (45211,)
True


## 5. Separação entre Treino e Teste

Nesta etapa, os dados são divididos em dois subconjuntos: **treino** e **teste**.

O conjunto de treino, composto por 80% dos dados, é utilizado para treinar os modelos de machine learning, enquanto o conjunto de teste, com 20% dos dados, é reservado para avaliar o desempenho do modelo em dados não vistos anteriormente.

Essa abordagem, conhecida como **holdout**, é fundamental para evitar *overfitting* e garantir que o modelo seja capaz de generalizar bem para novos dados.

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (36168, 42)
X_test: (9043, 42)


## 6. Normalização dos Dados

Nesta etapa, é realizada a normalização dos dados, com o objetivo de padronizar a escala das variáveis numéricas.

Algoritmos como **SVM** e **KNN** são sensíveis à escala dos dados, o que pode impactar diretamente o desempenho do modelo. Dessa forma, a normalização garante que todas as variáveis contribuam de maneira equilibrada para o processo de aprendizagem.

Neste projeto, a normalização será aplicada dentro de um **Pipeline**, assegurando que o mesmo processo utilizado durante o treinamento também seja aplicado automaticamente em novos dados durante a fase de predição no backend.

In [5]:
print("A normalização será feita dentro do Pipeline do modelo.")

A normalização será feita dentro do Pipeline do modelo.


## 7. Treinamento e Avaliação dos Modelos

Nesta etapa, são treinados e avaliados diferentes algoritmos de machine learning com o objetivo de comparar seus desempenhos na tarefa de classificação.

Os modelos utilizados foram:
- **K-Nearest Neighbors (KNN)**
- **Árvore de Decisão**
- **Naive Bayes**
- **Support Vector Machine (SVM)**

Como alguns algoritmos, como **KNN** e **SVM**, são sensíveis à escala dos dados, foi aplicada uma normalização utilizando o **StandardScaler** para garantir uma comparação justa entre os modelos.

A avaliação dos modelos foi realizada utilizando a métrica de **acurácia (accuracy)**, além do **classification report**, que apresenta métricas como precisão (*precision*), revocação (*recall*) e F1-score.

Essa abordagem permite analisar não apenas o desempenho geral dos modelos, mas também sua capacidade de classificação em cada classe.

In [6]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

# cópias escaladas apenas para avaliação comparativa
scaler_temp = StandardScaler()
X_train_scaled = scaler_temp.fit_transform(X_train)
X_test_scaled = scaler_temp.transform(X_test)

results = {}

# KNN
knn = KNeighborsClassifier()
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)
results["KNN"] = accuracy_score(y_test, y_pred_knn)

print("\n=== KNN ===")
print(f"Accuracy: {results['KNN']:.4f}")
print(classification_report(y_test, y_pred_knn))

# Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
results["Decision Tree"] = accuracy_score(y_test, y_pred_dt)

print("\n=== Decision Tree ===")
print(f"Accuracy: {results['Decision Tree']:.4f}")
print(classification_report(y_test, y_pred_dt))

# Naive Bayes
nb = GaussianNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)
results["Naive Bayes"] = accuracy_score(y_test, y_pred_nb)

print("\n=== Naive Bayes ===")
print(f"Accuracy: {results['Naive Bayes']:.4f}")
print(classification_report(y_test, y_pred_nb))

# SVM
svm = SVC()
svm.fit(X_train_scaled, y_train)
y_pred_svm = svm.predict(X_test_scaled)
results["SVM"] = accuracy_score(y_test, y_pred_svm)

print("\n=== SVM ===")
print(f"Accuracy: {results['SVM']:.4f}")
print(classification_report(y_test, y_pred_svm))


=== KNN ===
Accuracy: 0.8933
              precision    recall  f1-score   support

           0       0.91      0.97      0.94      7952
           1       0.60      0.33      0.43      1091

    accuracy                           0.89      9043
   macro avg       0.76      0.65      0.69      9043
weighted avg       0.88      0.89      0.88      9043


=== Decision Tree ===
Accuracy: 0.8705
              precision    recall  f1-score   support

           0       0.93      0.92      0.93      7952
           1       0.46      0.48      0.47      1091

    accuracy                           0.87      9043
   macro avg       0.70      0.70      0.70      9043
weighted avg       0.87      0.87      0.87      9043


=== Naive Bayes ===
Accuracy: 0.8578
              precision    recall  f1-score   support

           0       0.93      0.91      0.92      7952
           1       0.42      0.50      0.46      1091

    accuracy                           0.86      9043
   macro avg       0

## 8. Validação Cruzada (Cross-Validation)

Nesta etapa, é aplicada a técnica de **validação cruzada (cross-validation)** com o objetivo de avaliar a estabilidade e a capacidade de generalização dos modelos.

Nessa abordagem, os dados de treino são particionados em múltiplos subconjuntos (*folds*). O modelo é treinado em parte desses dados e validado em outro subconjunto, repetindo o processo diversas vezes.

Esse procedimento permite uma avaliação mais robusta do desempenho dos modelos, reduzindo a dependência de uma única divisão entre treino e teste.

Além disso, foram utilizados **pipelines** para os modelos que necessitam de normalização, garantindo que o pré-processamento seja aplicado corretamente dentro do processo de validação.

In [7]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

modelos_cv = {
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier())
    ]),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Naive Bayes": GaussianNB(),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC())
    ])
}

for name, model in modelos_cv.items():
    scores = cross_val_score(model, X_train, y_train, cv=5)
    print(f"{name} CV Accuracy: {scores.mean():.4f}")

KNN CV Accuracy: 0.8936
Decision Tree CV Accuracy: 0.8727
Naive Bayes CV Accuracy: 0.8637
SVM CV Accuracy: 0.9025


## 9. Otimização de Hiperparâmetros

Nesta etapa, foi realizada a otimização dos hiperparâmetros do modelo **Support Vector Machine (SVM)** utilizando a técnica de **Grid Search** com validação cruzada (*GridSearchCV*).

O objetivo é encontrar a melhor combinação de parâmetros que maximize o desempenho do modelo. Foram avaliados diferentes valores para o parâmetro **C**, responsável pelo controle da regularização, e diferentes tipos de **kernel**, que definem a forma como os dados são projetados no espaço.

A avaliação foi realizada por meio de **validação cruzada**, garantindo maior robustez na escolha dos melhores parâmetros e reduzindo o risco de overfitting.

Além disso, foi utilizado um **Pipeline**, assegurando que o processo de normalização dos dados seja aplicado corretamente em conjunto com o treinamento do modelo.

In [8]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC())
])

param_grid = {
    "svm__C": [0.1, 1],
    "svm__kernel": ["linear"]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=3,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Melhores parâmetros:", grid.best_params_)

best_model = grid.best_estimator_

Melhores parâmetros: {'svm__C': 0.1, 'svm__kernel': 'linear'}


## 10. Exportação do Modelo

Após a etapa de otimização dos hiperparâmetros, o melhor modelo encontrado é selecionado como modelo final do projeto.

Esse modelo é então avaliado no conjunto de teste, com o objetivo de verificar seu desempenho em dados não vistos durante o treinamento.

O modelo final será exportado e utilizado na aplicação **full stack**, permitindo que novos dados sejam inseridos no sistema e que previsões sejam realizadas em tempo real.

In [9]:
from sklearn.metrics import accuracy_score

y_pred = best_model.predict(X_test)
print("Accuracy final:", accuracy_score(y_test, y_pred))



Accuracy final: 0.8907442220502045


## 11. Exportação dos Artefatos

Nesta etapa, são exportados os artefatos gerados durante o treinamento do modelo, permitindo sua utilização posterior na aplicação backend.

O modelo final treinado é salvo em formato de arquivo, possibilitando seu carregamento sem a necessidade de novo treinamento. Além disso, também são armazenadas as colunas utilizadas no treinamento, garantindo consistência na entrada de dados durante a predição.

Essa abordagem é essencial para integração com a aplicação **full stack**, assegurando que o modelo receba dados no mesmo formato em que foi treinado.

In [10]:
import joblib
import os

# cria pasta correta
os.makedirs("./MachineLearning/models", exist_ok=True)

# salva modelo com nome esperado pelo backend
joblib.dump(best_model, "./MachineLearning/models/bank_marketing_svm.pkl")

# salva colunas
joblib.dump(X_train.columns.tolist(), "./MachineLearning/models/colunas_modelo.pkl")

print("Arquivos salvos corretamente para o backend!")

Arquivos salvos corretamente para o backend!


## 12. Análise Final

O modelo **SVM (Support Vector Machine)** apresentou o melhor desempenho entre os algoritmos avaliados, sendo selecionado como modelo final do projeto.

Durante a análise dos dados, observou-se que a variável **"duration"** possui grande impacto na predição. No entanto, sua utilização pode não ser adequada em cenários reais, uma vez que essa informação só é conhecida após o contato com o cliente, o que pode introduzir viés no modelo.

Como melhoria futura, recomenda-se avaliar o desempenho do modelo sem a utilização dessa variável, a fim de torná-lo mais aderente a um cenário real de negócio.

De modo geral, o modelo desenvolvido pode ser utilizado como apoio às estratégias de marketing, permitindo a priorização de clientes com maior probabilidade de adesão ao produto e contribuindo para a otimização de campanhas.